In [188]:
!pip install nba_api

In [189]:
import pandas as pd
import numpy as np

In [190]:
from nba_api.stats.endpoints import commonallplayers

player_info = commonallplayers.CommonAllPlayers()
dfs = player_info.get_data_frames()
dfs = dfs[0]
player_ids = dfs['PERSON_ID'].tolist()
player_ids

[76001,
 76002,
 76003,
 51,
 1505,
 949,
 76005,
 76006,
 76007,
 203518,
 1630173,
 101165,
 76008,
 76009,
 76010,
 203112,
 76011,
 76012,
 200801,
 1629121,
 203919,
 149,
 203500,
 912,
 1628389,
 1629061,
 76015,
 202399,
 201167,
 1630534,
 200772,
 76016,
 201336,
 76017,
 201582,
 1642351,
 76018,
 1631231,
 203006,
 1629152,
 202374,
 76019,
 76020,
 1630583,
 203128,
 202332,
 200746,
 76021,
 1626146,
 724,
 2042,
 76022,
 201570,
 1629734,
 1641725,
 1630234,
 2349,
 1629638,
 76024,
 1628959,
 76028,
 1628960,
 1628386,
 706,
 1628443,
 202730,
 76027,
 2124,
 76025,
 951,
 1641851,
 2754,
 76029,
 200984,
 76030,
 201165,
 308,
 1747,
 1824,
 1630631,
 680,
 732,
 202329,
 200811,
 76034,
 2365,
 2431,
 101187,
 202079,
 76035,
 76036,
 1507,
 76037,
 944,
 246,
 202341,
 76040,
 1626147,
 72,
 76041,
 203937,
 76042,
 76043,
 98,
 76045,
 76046,
 201583,
 1000,
 335,
 76048,
 101149,
 76049,
 1628387,
 76050,
 1512,
 203507,
 1628961,
 203648,
 2546,
 1630175,
 21,
 20

In [ ]:
from nba_api.stats.endpoints import playercareerstats
from nba_api.stats.static import players

# In this case it's LeBron James
player_id = 2544
# Get career stats
career = playercareerstats.PlayerCareerStats(player_id=player_id)
# Turn it into a pandas DataFrame for easy viewing
career_stats_df = career.get_data_frames()[0]
# Display
pd.options.display.max_columns = None
career_stats_df.head()

In [ ]:
# Script to reduce memory usage
def reduce_memory_usage(df):
    start_mem = df.memory_usage().sum() / 1024**2
    print(f'Memory usage of dataframe is {start_mem:.2f} MB')

    for col in df.columns:
        col_type = df[col].dtype

        if col_type != object:
            c_min = df[col].min()
            c_max = df[col].max()

            if str(col_type)[:3] == 'int':
                if c_min > -128 and c_max < 127:
                    df[col] = df[col].astype('int8')
                elif c_min > -32768 and c_max < 32767:
                    df[col] = df[col].astype('int16')
                elif c_min > -2147483648 and c_max < 2147483647:
                    df[col] = df[col].astype('int32')
                else:
                    df[col] = df[col].astype('int64')
            else:
                if c_min > -3.4e+38 and c_max < 3.4e+38:
                    df[col] = df[col].astype('float32')
                else:
                    df[col] = df[col].astype('float64')

    end_mem = df.memory_usage().sum() / 1024**2
    print(f'Memory usage after optimization is: {end_mem:.2f} MB')
    print(f'Decreased by {(start_mem - end_mem) / start_mem * 100:.1f}%')

reduce_memory_usage(career_stats_df)

In [ ]:
career_stats_df.dtypes

In [ ]:
cols_to_avoid = ['PLAYER_ID', 'SEASON_ID', 'LEAGUE_ID', 'TEAM_ID', 'PLAYER_AGE', 'GP', 'GS']
cols_to_divide = []
for col in career_stats_df.columns:
    if career_stats_df[col].dtype != 'float32' and career_stats_df[col].dtype != 'object' and col not in cols_to_avoid:
        cols_to_divide.append(col)
cols_to_divide

In [ ]:
games_played = career_stats_df['GP']
games_played

In [ ]:
for col in cols_to_divide:
    career_stats_df[col + '_PER_GAME'] = career_stats_df[col].div(career_stats_df['GP'], axis=0)
career_stats_df

In [ ]:
reduce_memory_usage(career_stats_df)

In [ ]:
from sklearn.preprocessing import LabelEncoder

LE = LabelEncoder()
categorical_columns = ['SEASON_ID', 'LEAGUE_ID', 'TEAM_ABBREVIATION']
for col in categorical_columns:
    career_stats_df[col] = LE.fit_transform(career_stats_df[col])

In [ ]:
# Let's start looking at how to preprocess the columns
career_stats_df.isnull().sum()

In [ ]:
# Let's split our data into training and validation since everything is sorted: no null values and no categorical variables
# and train our XGBoost regressor
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split

y = career_stats_df['PTS_PER_GAME']
career_stats_df.drop(columns=['PTS_PER_GAME'], inplace=True)
X = career_stats_df
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
model = XGBRegressor(colsample_bylevel=0.9,
                      learning_rate=0.2,
                      colsample_bytree=0.8,
                      gamma=0.1,
                      max_depth=10,
                      min_child_weight=1,
                      n_estimators=250,
                      nthread=4,
                      random_state=42,
                     )
model.fit(X_train, y_train)
valid_predictions = model.predict(X_valid)

In [ ]:
from sklearn.metrics import mean_squared_error

mean_squared_error(valid_predictions, y_valid)

In [ ]:
valid_predictions